# 🩸 تدريب موديل تحليل صورة الدم الكاملة (CBC)

**الخطوات (من غير ما تفهم أي كود):**
1. من القائمة فوق: **Runtime ← Run all** (مش محتاج GPU، التدريب بياخد دقيقة أو اتنين بس).
2. لو ظهرت رسالة تطلب تسجيل دخول Kaggle في الخلية التالتة: اتبع التعليمات (حساب Kaggle مجاني).
3. لو الخلية الأولى قالت **"اعمل Restart session"**: من القائمة **Runtime ← Restart session** ثم **Run all** تاني.
4. في الآخر هينزل لك ملف `cbc_results.zip` تلقائيًا.
5. فك الضغط وحط `cbc_model.joblib` و `model_meta.json` في فولدر `model` بتاع المشروع.
6. فيه كمان `project_report_ar.html` (تقرير عربي جاهز بالأرقام الحقيقية، افتحه في المتصفح واطبعه PDF) وصور الـ confusion matrix.

> الخلية الأولى بتثبّت إصدار scikit-learn بالظبط (1.6.1) عشان نفس الإصدار يشتغل في الموقع، ده كان سبب مشاكل في المشروع الأول.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.6.1"])
import sklearn
print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.6.1", "اعمل Runtime ← Restart session ثم Run all تاني (لازم الإصدار يبقى 1.6.1)"

## 1) تحميل الداتا من Kaggle

In [ ]:
import kagglehub

SLUG = "ehababoelnaga/anemia-types-classification"
try:
    path = kagglehub.dataset_download(SLUG)
except Exception as e:
    print("محتاج تسجيل دخول Kaggle:", e)
    kagglehub.login()   # هيظهر لك مربع تحط فيه اسم المستخدم والـ API key
    path = kagglehub.dataset_download(SLUG)

print("Dataset path:", path)

In [ ]:
import glob, os
import pandas as pd

df = None
for f in glob.glob(os.path.join(path, "**", "*.csv"), recursive=True):
    tmp = pd.read_csv(f)
    tmp.columns = [str(c).strip().upper() for c in tmp.columns]
    if "DIAGNOSIS" in tmp.columns:
        df = tmp
        print("استخدمنا الملف:", f)
        break
if df is None:
    raise FileNotFoundError("ما لقيتش ملف فيه عمود Diagnosis. شوف محتوى فولدر الداتا فوق.")

print("الشكل:", df.shape)
print(df["DIAGNOSIS"].value_counts())

## 2) اختيار الحالات الأربع وتنضيف الداتا

In [ ]:
import re
import numpy as np

# ⚠️ لازم تتطابق مع FEATURE_INFO في config.py (نفس الأسماء والوحدات)
FEATURES = ["HGB", "RBC", "HCT", "WBC", "MCV", "MCH", "MCHC", "PLT"]
HARD = {"HGB": (2, 25), "RBC": (0.5, 9), "HCT": (5, 70), "WBC": (0.1, 500),
        "MCV": (40, 150), "MCH": (10, 50), "MCHC": (20, 45), "PLT": (1, 2000)}

missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"أعمدة ناقصة في الداتا: {missing} — الأعمدة الموجودة: {list(df.columns)}")

def norm(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

LABEL_MAP = {
    "healthy": "healthy",
    "irondeficiencyanemia": "iron_deficiency",
    "macrocyticanemia": "macrocytic",
    "normocyticnormochromicanemia": "normocytic_normochromic",
}
df["label"] = df["DIAGNOSIS"].map(lambda s: LABEL_MAP.get(norm(s)))
found = set(df["label"].dropna().unique())
if found != set(LABEL_MAP.values()):
    raise ValueError(f"الحالات المتوقعة مش كلها موجودة. اللي لقيناه: {found}\nالأسماء في الداتا: {df['DIAGNOSIS'].unique()}")

data = df.dropna(subset=["label"])[FEATURES + ["label"]].copy()
data[FEATURES] = data[FEATURES].apply(pd.to_numeric, errors="coerce")

n0 = len(data)
data = data.dropna()
n1 = len(data)
data = data.drop_duplicates()          # التكرار بيخلي الاختبار "يشوف" صور من التدريب فيطلع أحسن من الحقيقة
n2 = len(data)
ok = np.ones(len(data), dtype=bool)
for f, (lo, hi) in HARD.items():
    ok &= data[f].between(lo, hi).values
data = data[ok].reset_index(drop=True)
print(f"قبل: {n0} | بعد حذف الفاضي: {n1} | بعد حذف المكرر: {n2} | بعد حذف القيم المستحيلة: {len(data)}")
print(data["label"].value_counts())

# فحص الوحدات: الوسيط لازم يكون في مدى منطقي، لو لأ يبقى الوحدات مختلفة عن اللي الموقع متوقعها
print()
for f, (lo, hi) in HARD.items():
    med = data[f].median()
    flag = "" if lo <= med <= hi else "   ⚠️⚠️ الوحدة غالبًا مختلفة! راجع قبل ما تكمل"
    print(f"{f:5s} الوسيط = {med:8.2f}{flag}")

## 3) التدريب (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

X = data[FEATURES]
y = data["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                               random_state=42, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv)
print(f"Cross-validation (5-fold) على بيانات التدريب: {cv_scores.mean() * 100:.2f}% ± {cv_scores.std() * 100:.2f}")

model.fit(X_train, y_train)
print("الحالات:", list(model.classes_))

## 4) تقييم الموديل على بيانات الاختبار (ماشافهاش قبل كده)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc * 100:.2f}%  (على {len(y_test)} عينة)")
report = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

imp = sorted(zip(FEATURES, model.feature_importances_), key=lambda kv: kv[1])
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh([k for k, _ in imp], [v for _, v in imp], color="#A4262C")
ax.set_title("Feature importance"); plt.tight_layout(); plt.savefig("feature_importance.png", dpi=150); plt.show()

## 5) حفظ الموديل والملفات + التقرير

In [ ]:
import base64, json, joblib, zipfile
from datetime import date

# أمثلة جاهزة للموقع: وسيط كل حالة (بنتأكد إن الموديل بيتعرف عليها)
examples = {c: {f: round(float(data.loc[data["label"] == c, f].median()), 2) for f in FEATURES}
            for c in model.classes_}
for c, ex in examples.items():
    p = model.predict(pd.DataFrame([ex], columns=FEATURES))[0]
    print(f"مثال {c:26s} -> الموديل قال: {p}", "" if p == c else "  ⚠️")

meta = {
    "features": FEATURES,
    "classes": list(model.classes_),
    "ranges": {f: [float(data[f].min()), float(data[f].max())] for f in FEATURES},
    "examples": examples,
    "sklearn_version": sklearn.__version__,
    "test_accuracy": float(test_acc),
    "n_samples": int(len(data)),
}
joblib.dump(model, "cbc_model.joblib", compress=3)
with open("model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

results = {"test_accuracy": float(test_acc), "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std()),
           "n_samples": int(len(data)), "class_counts": data["label"].value_counts().to_dict(),
           "sklearn_version": sklearn.__version__, "classification_report": report}
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ---- تقرير عربي جاهز بالأرقام الحقيقية ----
AR = {"healthy": "سليم", "iron_deficiency": "أنيميا نقص الحديد",
      "macrocytic": "أنيميا كبيرة الكريات", "normocytic_normochromic": "أنيميا طبيعية الكريات والصبغة"}
counts = data["label"].value_counts()
rows = "".join(
    f"<tr><td>{AR[c]}</td><td>{int(counts[c])}</td><td>{report[c]['precision'] * 100:.1f}%</td>"
    f"<td>{report[c]['recall'] * 100:.1f}%</td><td>{report[c]['f1-score'] * 100:.1f}%</td></tr>"
    for c in model.classes_)
cm_b64 = base64.b64encode(open("confusion_matrix.png", "rb").read()).decode()
html = '''<!doctype html><html lang="ar" dir="rtl"><head><meta charset="utf-8"><title>تقرير المشروع</title>
<style>body{font-family:Tahoma,Arial,sans-serif;max-width:800px;margin:2rem auto;line-height:2;color:#12303B;padding:0 1rem}
h1{border-bottom:3px solid #A4262C;padding-bottom:.4rem}h2{margin-top:1.8rem;color:#A4262C}
table{border-collapse:collapse;width:100%}td,th{border:1px solid #bbb;padding:.4rem .7rem;text-align:center}th{background:#eef3f5}
.warn{background:#fff6e5;border:1px solid #f0d8a8;padding:.7rem 1rem;border-radius:8px}</style></head><body>
<h1>تقرير مشروع: محلّل صورة الدم الكاملة (CBC) بالذكاء الاصطناعي</h1>
<p>إعداد: أحمد عبدالجيد &nbsp;|&nbsp; كلية تكنولوجيا العلوم الصحية التطبيقية، قسم التحاليل &nbsp;|&nbsp; التاريخ: __DATE__</p>
<h2>1) فكرة المشروع</h2>
<p>موقع بياخد قيم تحليل صورة الدم الكاملة (CBC) من المستخدم، ويستخدم نموذج تعلّم آلة لتقدير أقرب نمط من 4 حالات، ويعرض مقارنة القيم بالمدى الطبيعي ومعلومات مبسطة عن الحالة.</p>
<h2>2) البيانات</h2>
<p>مجموعة بيانات Anemia Types Classification من Kaggle. اخترنا 4 حالات: سليم، أنيميا نقص الحديد، أنيميا كبيرة الكريات، أنيميا طبيعية الكريات والصبغة. استخدمنا 8 قياسات: HGB, RBC, HCT, WBC, MCV, MCH, MCHC, PLT. بعد حذف القيم الناقصة والمكررة والمستحيلة بقي <b>__N__</b> عينة.</p>
<h2>3) النموذج</h2>
<p>Random Forest (200 شجرة قرار) مع موازنة أوزان الحالات. قسمنا البيانات 80% تدريب و20% اختبار بنفس نسب الحالات، وقسم الاختبار لم يرَه النموذج أثناء التدريب.</p>
<h2>4) النتائج</h2>
<p>دقة الاختبار: <b>__ACC__%</b> &nbsp;|&nbsp; دقة Cross-validation (5 طيّات) على بيانات التدريب: <b>__CV__%</b></p>
<table><tr><th>الحالة</th><th>عدد العينات</th><th>Precision</th><th>Recall</th><th>F1</th></tr>__ROWS__</table>
<p style="text-align:center"><img src="data:image/png;base64,__CM__" width="380" alt="confusion matrix"><br>مصفوفة الالتباس (Confusion Matrix)</p>
<h2>5) الحدود</h2>
<ul><li>نسخة أولية تغطي 4 حالات فقط ولا تعرف أمراض الدم الأخرى.</li>
<li>تحليل CBC وحده لا يكفي للتشخيص، والطبيب يربطه بالأعراض وتحاليل أخرى.</li>
<li>البيانات من مصدر واحد، فالنتائج قد تختلف مع أجهزة ومعامل أخرى.</li></ul>
<p class="warn">⚠️ المشروع تعليمي فقط، والنتيجة ليست تشخيصًا طبيًا ولا بديلًا عن الطبيب.</p>
</body></html>'''
for k, v in {"__DATE__": date.today().isoformat(), "__N__": str(len(data)), "__ACC__": f"{test_acc * 100:.1f}",
             "__CV__": f"{cv_scores.mean() * 100:.1f}", "__ROWS__": rows, "__CM__": cm_b64}.items():
    html = html.replace(k, v)
with open("project_report_ar.html", "w", encoding="utf-8") as f:
    f.write(html)

with zipfile.ZipFile("cbc_results.zip", "w") as z:
    for name in ["cbc_model.joblib", "model_meta.json", "results.json", "project_report_ar.html",
                 "confusion_matrix.png", "feature_importance.png"]:
        z.write(name)

print(f"✅ خلص! دقة الموديل على الاختبار: {test_acc * 100:.2f}%")
try:
    from google.colab import files
    files.download("cbc_results.zip")
except ImportError:
    pass